# *Nonlinear Arterial Hemodynamics*
## Chapter 6 companion — Wall Compliance and Mean Transport

This notebook is the computational companion to Chapter 6. It reconstructs the finite-$k$ compliant first-order mode, the $O(\epsilon^2)$ Reynolds-stress forcing, the Taylor-expanded moving-wall closure, and the resulting steady-streaming observables.

The book nomenclature governs all reader-facing quantities.

The notebook admits the Chapter 6 mechanics only: a straight circular tube, scalar Newtonian fluid, thin Kelvin--Voigt wall, small radial displacement, finite-$k$ traveling-wave kinematics, second-order mean transport, and only the higher-order envelope material needed to locate the perturbation hierarchy.

Complex vascular geometry, constitutive anisotropy, and finite-amplitude transition are withheld.

**Execution:** a clean Google Colab runtime should reproduce all outputs with **Run all** and no manual parameter choices.

### Chapter question

Chapter 6 asks whether wall compliance can convert a zero-mean oscillatory mode into persistent time-averaged transport through nonlinear self-interaction.

The notebook therefore has five responsibilities:

1. reproduce the finite-$k$ compliant first-order field;
2. verify the canonical moving-wall Case C benchmark from Chapter 8;
3. construct the zero-frequency Reynolds-stress forcing and solve the second-order mean problem;
4. separate mean flux from moving-interface mean traction;
5. use VascuQuest to characterize physiological wall-motion amplitude and pressure--area phase without pretending that local waveform data uniquely identify the Kelvin--Voigt coefficients.

The perturbation order remains explicit throughout:

$$
O(\epsilon): \text{oscillatory mode},
$$

$$
O(\epsilon^2): \text{mean field and second harmonic},
$$

$$
O(\epsilon^3): \text{envelope solvability}.
$$

### VascuQuest representation

VascuQuest/PWDB supplies:

- age;
- heart rate;
- pressure waveform;
- flow-velocity waveform;
- luminal-area waveform.

The source flow-velocity and luminal-area signals are combined internally to recover

$$
Q(t)=U(t)A(t).
$$

The area waveform is converted to an equivalent circular radius,

$$
R(t)=\sqrt{\frac{A(t)}{\pi}},
$$

with reference radius

$$
R=\langle R(t)\rangle_t
$$

and observed relative wall-motion amplitude

$$
\epsilon_{\mathrm{VQ}}
=
\frac{\max_t|R(t)-R|}{R}.
$$

This quantity is a direct kinematic descriptor of the virtual population. It is not identified with a calibrated Kelvin--Voigt coefficient.

The local first-harmonic pressure--area phase is also computed descriptively. It documents the phase relation present in PWDB but does not uniquely determine

$$
K_s,\quad \tau_v,\quad T_w,\quad \rho_w h_w.
$$

When the notebook explores those wall coefficients, they are introduced as explicit controlled parameters through

$$
Z_w(\Omega,k)
=
K_s(1-i\Omega\tau_v)
+
T_wk^2
-
\rho_wh_w\Omega^2,
$$

not inferred silently from the database.

In [ ]:
# Configuration and reproducibility constants
from pathlib import Path
import sys, json, subprocess, zipfile

ROOT = Path("/content/nonlinear_arterial_hemodynamics_ch06")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"
META_DIR = ROOT / "metadata"
for d in (ROOT, FIG_DIR, DATA_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

VQ_REPOSITORY = "https://github.com/KNOWDYN/VascuQuest.git"
VQ_GIT_REF = "8307147d72e7a6f3ea3135895bd6f52927c67439"
PWDB_RECORD_ID = "3275625"
PWDB_DOI = "10.5281/zenodo.3275625"

rho = 1060.0       # kg m^-3
mu = 3.5e-3        # Pa s
nu = mu / rho      # m^2 s^-1

# Chapter 8 Case C benchmark
R_caseC = 4e-3
f_caseC = 1.2
Omega_caseC = 2.0 * 3.141592653589793 * f_caseC
kR_caseC = 0.2
k_caseC = kR_caseC / R_caseC
P_caseC = 1.0

SITES = [
    "AorticRoot", "ThorAorta", "AbdAorta",
    "Carotid", "Brachial", "Radial", "Femoral",
]

print("Working directory:", ROOT)
print(f"nu = {nu:.6e} m^2/s")

In [ ]:
# Install the pinned VascuQuest revision.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+{VQ_REPOSITORY}@{VQ_GIT_REF}"
])

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.special import jv, iv
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve
import vascuquest as vq

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("VascuQuest:", getattr(vq, "__version__", "version field not exposed"))

In [ ]:
# Acquire and verify the PWDB artifacts used by Chapter 6.
ARTIFACTS = [
    "model_configurations",
    "common_site_waveforms_csv",
]
verification = {}

for artifact in ARTIFACTS:
    subprocess.run(
        ["vascuquest", "dataset", "acquire",
         "--artifact", artifact, "--yes", "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verified = subprocess.run(
        ["vascuquest", "dataset", "verify",
         "--artifact", artifact, "--format", "json"],
        check=True, text=True, capture_output=True,
    )
    verification[artifact] = json.loads(verified.stdout)

status = subprocess.run(
    ["vascuquest", "dataset", "status", "--format", "json"],
    check=True, text=True, capture_output=True,
)
dataset_status = json.loads(status.stdout)
SOURCE_DIR = Path(dataset_status["managed_paths"]["source"])

(META_DIR / "artifact_verification.json").write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
(META_DIR / "dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print("Verified PWDB source:", SOURCE_DIR)

In [ ]:
# Open the verified dataset and establish deterministic subject metadata.
session = vq.open_dataset(source=SOURCE_DIR, offline=True)
assert session.identity.record_id == PWDB_RECORD_ID

age_result = session.get("age")
subject_ids = np.asarray(age_result.coordinates[0].values, dtype=str)
ages = np.asarray(age_result.values, dtype=float)

hr_result = session.get("heart_rate", subjects=subject_ids.tolist())
hr_ids = np.asarray(hr_result.coordinates[0].values, dtype=str)
heart_rates = np.asarray(hr_result.values, dtype=float)
assert np.array_equal(subject_ids, hr_ids)

subject_meta = pd.DataFrame({
    "subject_id": subject_ids,
    "age_years": ages,
    "heart_rate_bpm": heart_rates,
})
subject_meta["subject_number"] = subject_meta["subject_id"].astype(int)
subject_meta = subject_meta.sort_values("subject_number").reset_index(drop=True)

source_ages = sorted(subject_meta["age_years"].dropna().unique())
target_age = source_ages[len(source_ages)//2]
age_group = subject_meta.loc[subject_meta["age_years"] == target_age].copy()
age_group = age_group.sort_values("subject_number").reset_index(drop=True)
representative_subject = str(age_group.iloc[len(age_group)//2]["subject_id"])

selection_record = {
    "rule": "middle PWDB source age stratum; median canonical subject number",
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "source_age_strata_years": [float(x) for x in source_ages],
}
(META_DIR / "representative_subject.json").write_text(
    json.dumps(selection_record, indent=2), encoding="utf-8"
)
display(pd.DataFrame([selection_record]))

In [ ]:
# Shared B&W plotting system and PWDB waveform reader.
WAVE_ZIP = SOURCE_DIR / "PWs_csv.zip"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 9.0,
    "axes.titlesize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 7.8,
    "axes.linewidth": 0.75,
    "lines.linewidth": 1.2,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
BLACK, DARK, MID, LIGHT = "0.0", "0.28", "0.52", "0.74"

SITE_LABELS = {
    "AorticRoot": "Aortic root",
    "ThorAorta": "Thoracic aorta",
    "AbdAorta": "Abdominal aorta",
    "Carotid": "Carotid",
    "Brachial": "Brachial",
    "Radial": "Radial",
    "Femoral": "Femoral",
}

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def save_figure(fig, stem):
    pdf = FIG_DIR / f"{stem}.pdf"
    png = FIG_DIR / f"{stem}.png"
    fig.savefig(pdf, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(png, dpi=600, bbox_inches="tight", pad_inches=0.03)
    return pdf, png

def _wave_member_name(site_id, source_signal):
    basename = f"PWs_{site_id}_{source_signal}.csv"
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        matches = [name for name in zf.namelist() if Path(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {basename!r}; found {len(matches)}")
    return matches[0]

def load_waveform_matrix(site_id, source_signal):
    # Database-native signal labels stay in this mapping layer.
    member = _wave_member_name(site_id, source_signal)
    with zipfile.ZipFile(WAVE_ZIP, "r") as zf:
        with zf.open(member, "r") as raw:
            frame = pd.read_csv(raw, low_memory=False)
    ids = np.asarray([str(int(x)) for x in frame.iloc[:, 0].to_numpy()], dtype=str)
    values = frame.iloc[:, 1:].to_numpy(dtype=float)
    return ids, values


def active_prefix(row):
    """Return the finite leading waveform block; reject internal missing samples."""
    row = np.asarray(row, dtype=float)
    finite = np.isfinite(row)
    if not finite.any():
        return None
    last = np.flatnonzero(finite)[-1] + 1
    y = row[:last]
    if np.any(~np.isfinite(y)):
        return None
    return y

def first_harmonic(y):
    """Complex first Fourier-series coefficient using the notebook phase convention."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n < 8:
        return np.nan + 1j*np.nan
    j = np.arange(n, dtype=float)
    return np.sum(y*np.exp(-2j*np.pi*j/n))/n


def phase_aware_median(angles):
    """Median angle after recentering on the circular mean to avoid +/-pi wrap artifacts."""
    angles = np.asarray(angles, dtype=float)
    angles = angles[np.isfinite(angles)]
    if len(angles) == 0:
        return np.nan
    center = np.angle(np.mean(np.exp(1j*angles)))
    local = center + np.angle(np.exp(1j*(angles-center)))
    return np.angle(np.exp(1j*np.median(local)))

print("Shared helpers ready.")

# Small wall motion as the organizing parameter

The instantaneous wall position is

$$
r=R+\eta(z,t),
$$

with

$$
\epsilon=\frac{\max|\eta|}{R}\ll1.
$$

The perturbation hierarchy is

$$
\mathbf u
=
\epsilon\mathbf u_1
+
\epsilon^2\mathbf u_2
+
O(\epsilon^3),
$$

$$
p
=
p_0
+
\epsilon p_1
+
\epsilon^2p_2
+
O(\epsilon^3),
$$

$$
\eta
=
\epsilon\eta_1
+
\epsilon^2\eta_2
+
O(\epsilon^3).
$$

The base state is quiescent,

$$
\nabla p_0=0,
\qquad
\mathbf u_0=0.
$$

This ordering fixes the interpretation: the first-order response is oscillatory and has zero period mean; the mean transport first appears at $O(\epsilon^2)$.

# First-order compliant mode

At $O(\epsilon)$,

$$
\rho\frac{\partial\mathbf u_1}{\partial t}
=
-\nabla p_1
+
\mu\nabla^2\mathbf u_1,
\qquad
\nabla\cdot\mathbf u_1=0.
$$

The model is axisymmetric and non-swirling,

$$
\mathbf u_1
=
u_{1r}\mathbf e_r
+
u_{1z}\mathbf e_z.
$$

For traveling dependence

$$
e^{i(kz-\Omega t)},
$$

the regular pressure field is

$$
\widehat p(r)=P I_0(kr).
$$

Define

$$
\lambda_C^2
=
i\frac{\rho\Omega}{\mu}
-
k^2.
$$

The regular first-order velocity representation is

$$
\widehat u_z
=
\frac{kP}{\Omega\rho}I_0(kr)
+
C_\psi\lambda_CJ_0(\lambda_Cr),
$$

$$
\widehat u_r
=
-i\left[
\frac{kP}{\Omega\rho}I_1(kr)
+
kC_\psi J_1(\lambda_Cr)
\right].
$$

Axial no slip determines $C_\psi$, and kinematic compatibility then determines $\widehat\eta$ for the prescribed traveling mode.

In [ ]:
# Exact first-order finite-k compliant/Stokes representation.
def compliant_first_order(
    r,
    R,
    Omega,
    k,
    P,
    rho=rho,
    mu=mu,
):
    r = np.asarray(r, dtype=float)
    lambda_C = np.sqrt(1j*rho*Omega/mu - k**2)
    A = k*P/(Omega*rho)

    C_psi = (
        -A*iv(0, k*R)
        /(lambda_C*jv(0, lambda_C*R))
    )

    uz = A*iv(0, k*r) + C_psi*lambda_C*jv(0, lambda_C*r)
    ur = -1j*(
        A*iv(1, k*r)
        + k*C_psi*jv(1, lambda_C*r)
    )

    uz_r = (
        A*k*iv(1, k*r)
        - C_psi*lambda_C**2*jv(1, lambda_C*r)
    )

    ur_r = -1j*(
        A*k*0.5*(iv(0, k*r)+iv(2, k*r))
        + k*C_psi*lambda_C*0.5*(
            jv(0, lambda_C*r)-jv(2, lambda_C*r)
        )
    )

    eta_hat = (
        A*iv(1, k*R)
        + k*C_psi*jv(1, lambda_C*R)
    ) / Omega

    p = P*iv(0, k*r)

    return {
        "lambda_C": lambda_C,
        "C_psi": C_psi,
        "p": p,
        "uz": uz,
        "ur": ur,
        "uz_r": uz_r,
        "ur_r": ur_r,
        "eta_hat": eta_hat,
    }

def compatible_wall_impedance(
    R, Omega, k, P, rho=rho, mu=mu
):
    fields = compliant_first_order(
        np.array([R]), R, Omega, k, P, rho, mu
    )
    eta_hat = fields["eta_hat"]
    pR = fields["p"][0]
    ur_r_R = fields["ur_r"][0]
    Z_compat = (pR - 2.0*mu*ur_r_R)/eta_hat
    return Z_compat

print("First-order compliant-mode functions ready.")

# Boundary conditions and wall impedance

At the wall,

$$
u_{1z}(R,z,t)=0,
$$

$$
u_{1r}(R,z,t)
=
\frac{\partial\eta_1}{\partial t}.
$$

The thin-wall balance is

$$
\rho_wh_w\eta_{1,tt}
+
K_s\eta_1
+
K_s\tau_v\eta_{1,t}
-
T_w\eta_{1,zz}
=
p_1
-
2\mu\partial_r u_{1r}.
$$

Therefore,

$$
Z_w(\Omega,k)
=
K_s(1-i\Omega\tau_v)
+
T_wk^2
-
\rho_wh_w\Omega^2.
$$

The notebook exposes this law directly rather than assigning physiological values to its coefficients without evidence.

In [ ]:
# Explicit coefficient parameterization for controlled wall-law studies.
def wall_impedance(
    Omega,
    k,
    K_s,
    tau_v=0.0,
    T_w=0.0,
    rho_w_h_w=0.0,
):
    return (
        K_s*(1.0 - 1j*Omega*tau_v)
        + T_w*k**2
        - rho_w_h_w*Omega**2
    )

def dimensionless_wall_impedance(
    R,
    Omega,
    k,
    K_s,
    tau_v=0.0,
    T_w=0.0,
    rho_w_h_w=0.0,
    mu=mu,
):
    return (
        R*wall_impedance(
            Omega, k, K_s, tau_v, T_w, rho_w_h_w
        )/(mu*Omega)
    )

print("Parameterized wall-impedance function ready.")

The function above is intentionally generic. It permits controlled studies of stiffness, viscoelastic phase lag, axial tension, and wall inertia while preserving the exact Chapter 6 sign convention.

No choice of $K_s$, $\tau_v$, $T_w$, or $\rho_wh_w$ is called physiological unless an independent measurement definition and source are supplied.

# Canonical Case C verification

Chapter 8 specifies

$$
\rho=1060\ {\rm kg\,m^{-3}},
\qquad
\mu=3.5\times10^{-3}\ {\rm Pa\,s},
$$

$$
R=4\ {\rm mm},
\qquad
f=1.2\ {\rm Hz},
\qquad
kR=0.2,
$$

and normalizes the finite-$k$ pressure amplitude to

$$
P=1\ {\rm Pa}.
$$

The reference values are

$$
\alpha=6.0445,
$$

$$
\widehat\eta
=
(6.3690-1.7230i)\times10^{-5}\ {\rm m},
$$

and

$$
Z_{\mathrm{compat}}
=
(1.4777+0.3984i)\times10^4\ {\rm Pa\,m^{-1}}.
$$

The compatible load ratio is a diagnostic for the prescribed traveling mode, not a calibrated arterial wall law.

In [ ]:
# Reproduce the Case C first-order quantities.
alpha_caseC = R_caseC*np.sqrt(rho*Omega_caseC/mu)

r_caseC = np.linspace(0.0, R_caseC, 1200)
fields_caseC = compliant_first_order(
    r_caseC,
    R_caseC,
    Omega_caseC,
    k_caseC,
    P_caseC,
)
eta_caseC = fields_caseC["eta_hat"]
Zcompat_caseC = compatible_wall_impedance(
    R_caseC,
    Omega_caseC,
    k_caseC,
    P_caseC,
)

caseC_first_order = pd.DataFrame([{
    "alpha": alpha_caseC,
    "eta_real_m": eta_caseC.real,
    "eta_imag_m": eta_caseC.imag,
    "Zcompat_real_Pa_per_m": Zcompat_caseC.real,
    "Zcompat_imag_Pa_per_m": Zcompat_caseC.imag,
}])
caseC_first_order.to_csv(
    DATA_DIR / "ch06_caseC_first_order.csv", index=False
)
display(caseC_first_order)

# Self-check against the Chapter 8 reference values.
eta_book = (6.3690 - 1.7230j)*1e-5
Zcompat_book = (1.4777 + 0.3984j)*1e4
assert abs(eta_caseC-eta_book)/abs(eta_book) < 5e-5
assert abs(Zcompat_caseC-Zcompat_book)/abs(Zcompat_book) < 5e-5


In [ ]:
# Visualize the radial structure of the canonical first-order mode.
uz = fields_caseC["uz"]
ur = fields_caseC["ur"]

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.05))

axes[0].plot(
    r_caseC/R_caseC,
    np.abs(uz)/np.max(np.abs(uz)),
    color=BLACK
)
axes[0].set_xlabel(r"Normalized radius, $r/R$")
axes[0].set_ylabel(r"$|\widehat u_z|/\max|\widehat u_z|$")
clean_axes(axes[0])

axes[1].plot(
    r_caseC/R_caseC,
    np.abs(ur)/np.max(np.abs(ur)),
    color=BLACK
)
axes[1].set_xlabel(r"Normalized radius, $r/R$")
axes[1].set_ylabel(r"$|\widehat u_r|/\max|\widehat u_r|$")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.4)
save_figure(fig, "ch06_caseC_first_order_mode")
plt.show()

The finite-$k$ mode contains radial motion as well as axial motion. That is the kinematic change introduced by the moving wall. It is already present at $O(\epsilon)$, even though the period mean remains zero.

# Rigid and long-wave recovery

The compliant finite-$k$ formulation must recover the rigid Womersley solution in the combined limit

$$
|\widetilde Z_w|\rightarrow\infty,
\qquad
k\rightarrow0,
$$

while

$$
\widehat G=-ikP
$$

is held finite.

The next calculation checks the finite-$k$ Stokes field against the rigid Womersley profile as $kR$ decreases.

In [ ]:
# Long-wave recovery at fixed axial pressure-gradient amplitude.
def womersley_e_minus_iwt(r, R, Omega, G, rho=rho, mu=mu):
    lambda_W = np.sqrt(1j*rho*Omega/mu)
    return (
        -G/(1j*Omega*rho)
        * (
            1.0
            - jv(0, lambda_W*r)/jv(0, lambda_W*R)
        )
    )

kR_values = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
G_hold = 1.0
recovery_rows = []

for khat in kR_values:
    kval = khat/R_caseC
    Pval = 1j*G_hold/kval   # -i k P = G
    rr = np.linspace(0.0, R_caseC, 900)

    finite = compliant_first_order(
        rr, R_caseC, Omega_caseC, kval, Pval
    )["uz"]
    rigid = womersley_e_minus_iwt(
        rr, R_caseC, Omega_caseC, G_hold
    )

    rel_error = (
        np.max(np.abs(finite-rigid))
        / np.max(np.abs(rigid))
    )
    recovery_rows.append({
        "kR": khat,
        "relative_profile_error": rel_error,
    })

recovery_df = pd.DataFrame(recovery_rows)
recovery_df.to_csv(
    DATA_DIR / "ch06_long_wave_recovery.csv", index=False
)

fig, ax = plt.subplots(figsize=(5.8, 3.2))
ax.loglog(
    recovery_df["kR"],
    recovery_df["relative_profile_error"],
    color=BLACK, marker="o", markersize=4
)
ax.set_xlabel(r"Dimensionless axial wavenumber, $\widehat k=kR$")
ax.set_ylabel("Relative axial-profile error")
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_long_wave_womersley_recovery")
plt.show()

The error decreases as $\widehat k\to0$, confirming that the finite-$k$ formulation approaches the classical rigid Womersley profile when the axial pressure-gradient amplitude is held fixed.

This is a limiting-case verification, not a claim that compliant arterial waves operate in the long-wave limit at every site.

# Second-order mean transport

Period averaging is

$$
\langle f\rangle
=
\frac{\Omega}{2\pi}
\int_0^{2\pi/\Omega}f(t)\,dt.
$$

Although

$$
\langle\mathbf u_1\rangle=0,
$$

the quadratic interaction has a nonzero zero-frequency component,

$$
\left\langle
(\mathbf u_1\cdot\nabla)\mathbf u_1
\right\rangle
=
\frac12
\Re\left\{
\overline{\widehat{\mathbf u}_1}
\cdot
\nabla\widehat{\mathbf u}_1
\right\}.
$$

The mean equation is

$$
\mu\nabla^2\langle\mathbf u_2\rangle
-
\nabla\langle p_2\rangle
=
\rho
\left\langle
(\mathbf u_1\cdot\nabla)\mathbf u_1
\right\rangle.
$$

For the Case C benchmark, no external $O(\epsilon^2)$ pressure drop is imposed.

# Moving-wall closure at second order

The axial no-slip condition is imposed at the moving wall,

$$
u_z(R+\eta,z,t)=0.
$$

Taylor expansion gives

$$
u_{2z}(R,z,t)
=
-\eta_1(z,t)
\left.
\frac{\partial u_{1z}}{\partial r}
\right|_R.
$$

Therefore,

$$
\left\langle u_{2z}(R)\right\rangle
=
-
\left\langle
\eta_1
\left.
\frac{\partial u_{1z}}{\partial r}
\right|_R
\right\rangle.
$$

This generally produces an inhomogeneous mean boundary condition. Replacing it with homogeneous no slip would remove a term of the same perturbation order as the Reynolds-stress forcing.

In [ ]:
# Case C second-order forcing and moving-wall boundary value.
eta_hat = fields_caseC["eta_hat"]
uz_hat = fields_caseC["uz"]
ur_hat = fields_caseC["ur"]
uz_r_hat = fields_caseC["uz_r"]

mean_convective_z = 0.5*np.real(
    np.conj(ur_hat)*uz_r_hat
    + np.conj(uz_hat)*(1j*k_caseC*uz_hat)
)

u2_wall = -0.5*np.real(
    np.conj(eta_hat)*uz_r_hat[-1]
)

print(
    "<u_2z(R)> =",
    f"{u2_wall:.8e}",
    "m/s"
)
print(
    "Chapter 8 reference = 2.46558e-4 m/s"
)
assert abs(u2_wall-2.46558e-4)/2.46558e-4 < 5e-6


### Numerical verification path used in this notebook

Chapter 8 states that the canonical Case C table was generated with a fourth-order finite-difference discretization. The compact solver below is deliberately an **independent** verification path: it uses a second-order centered radial operator with analytic centerline regularity and a fourth-order backward derivative for the wall-gradient contribution.

Accordingly, the notebook requires convergence to the Chapter 8 observables but does not claim that the coarse-grid sequence must be identical. The reference table is stored alongside the independently computed sequence and relative errors are reported explicitly.

In [ ]:
# Independent uniform-grid mean-flow solve for Case C.
# The interior radial operator is second-order centered; the wall derivative used
# for traction is fourth-order backward. This is intentionally treated as an
# independent convergence check, not as a reimplementation of the Chapter 8
# fourth-order finite-difference stencil.
def solve_streaming_case(
    N_s,
    R,
    Omega,
    k,
    P,
    rho=rho,
    mu=mu,
):
    r = np.linspace(0.0, R, N_s)
    dr = r[1]-r[0]

    f1 = compliant_first_order(
        r, R, Omega, k, P, rho, mu
    )

    uz = f1["uz"]
    ur = f1["ur"]
    uz_r = f1["uz_r"]
    eta_hat = f1["eta_hat"]

    mean_convective_z = 0.5*np.real(
        np.conj(ur)*uz_r
        + np.conj(uz)*(1j*k*uz)
    )

    rhs = rho*mean_convective_z/mu

    # Axisymmetric radial Laplacian:
    # u'' + (1/r)u'.
    A = lil_matrix((N_s, N_s), dtype=float)

    # Analytic regularity limit at r=0:
    # L u(0) = 4 (u_1-u_0)/dr^2 for a regular even field.
    A[0, 0] = -4.0/dr**2
    A[0, 1] = +4.0/dr**2

    for i in range(1, N_s-1):
        ri = r[i]
        A[i, i-1] = 1.0/dr**2 - 1.0/(2.0*ri*dr)
        A[i, i]   = -2.0/dr**2
        A[i, i+1] = 1.0/dr**2 + 1.0/(2.0*ri*dr)

    u2_wall = -0.5*np.real(
        np.conj(eta_hat)*uz_r[-1]
    )

    # Inhomogeneous Taylor-expanded moving-wall row.
    A[-1, -1] = 1.0
    rhs[-1] = u2_wall

    u2 = spsolve(A.tocsr(), rhs)

    Q_stream = 2.0*np.pi*np.trapezoid(
        r*u2, r
    )

    # Fourth-order backward derivative at the wall.
    u2_r_wall = (
        25.0*u2[-1]
        - 48.0*u2[-2]
        + 36.0*u2[-3]
        - 16.0*u2[-4]
        + 3.0*u2[-5]
    )/(12.0*dr)

    return {
        "r": r,
        "u2": u2,
        "Q_stream": Q_stream,
        "u2_wall": u2_wall,
        "u2_r_wall": u2_r_wall,
        "first_order": f1,
        "mean_convective_z": mean_convective_z,
    }

print("Second-order radial solver ready.")

In [ ]:
# Compare the independent radial refinement with the Chapter 8 Case C table.
Ns_values = [500, 1000, 2000, 4000]
book_Q = {
    500: 7.568807e-9,
    1000: 7.568655e-9,
    2000: 7.568617e-9,
    4000: 7.568608e-9,
}
book_tau = {
    500: -2.186351e-4,
    1000: -2.186217e-4,
    2000: -2.186183e-4,
    4000: -2.186175e-4,
}

streaming_rows = []
streaming_solutions = {}

for Ns in Ns_values:
    sol = solve_streaming_case(
        Ns,
        R_caseC,
        Omega_caseC,
        k_caseC,
        P_caseC,
    )
    streaming_solutions[Ns] = sol

    # Traction decomposition is evaluated after the helper is defined below;
    # here store the mean-flow quantities and benchmark errors.
    q_rel_err = abs(sol["Q_stream"] - book_Q[Ns]) / abs(book_Q[Ns])

    streaming_rows.append({
        "N_s": Ns,
        "Q_stream_m3_s": sol["Q_stream"],
        "book_Q_stream_m3_s": book_Q[Ns],
        "Q_relative_error": q_rel_err,
        "reference_radius_gradient_Pa": -mu*sol["u2_r_wall"],
    })

streaming_df = pd.DataFrame(streaming_rows)
streaming_df.to_csv(
    DATA_DIR / "ch06_caseC_streaming_refinement.csv",
    index=False
)
display(streaming_df)

# The independent scheme should converge tightly to the reported Case C flux.
assert streaming_df.iloc[-1]["Q_relative_error"] < 2e-6

The computed flux is compared directly with the Chapter 8 Case C sequence. Because this notebook uses an independent second-order interior radial operator rather than the book's stated fourth-order stencil, the coarse-grid values need not coincide. The acceptance criterion is convergence to the same reported observable; the finest-grid relative flux error is required to be below the explicit tolerance in the code.

The complete moving-interface traction is checked separately after all three $O(\epsilon^2)$ traction terms have been assembled.

In [ ]:
# Plot the converged Case C streaming profile.
stream4000 = streaming_solutions[4000]

fig, ax = plt.subplots(figsize=(5.8, 3.3))
ax.plot(
    stream4000["r"]/R_caseC,
    stream4000["u2"],
    color=BLACK
)
ax.scatter(
    [1.0],
    [stream4000["u2_wall"]],
    facecolors="white",
    edgecolors=BLACK,
    s=30,
    zorder=3,
    label=r"moving-wall boundary value"
)
ax.set_xlabel(r"Normalized radius, $r/R$")
ax.set_ylabel(r"$\langle u_{2z}\rangle$ (m s$^{-1}$)")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_streaming_profile_reproduced")
plt.show()

This reproduces the principal Chapter 6 book figure mechanically: the mean profile is not constrained to vanish at the reference wall because the second-order boundary condition is evaluated by Taylor expansion at the moving interface.

# Mean wall traction on the moving interface

The $O(\epsilon^2)$ mean tangential traction is

$$
\begin{aligned}
\langle\tau_w^{(2)}\rangle
={}&
\left\langle\sigma_{zr}^{(2)}(R)\right\rangle
+
\left\langle
\eta_1
\partial_r\sigma_{zr}^{(1)}(R)
\right\rangle\\
&+
\left\langle
(\partial_z\eta_1)
[
\sigma_{rr}^{(1)}(R)
-
\sigma_{zz}^{(1)}(R)
]
\right\rangle.
\end{aligned}
$$

The three contributions must be retained at the same perturbation order.

In [ ]:
# Exact first-order derivatives needed for the moving-surface traction correction.
def caseC_traction_terms(
    R,
    Omega,
    k,
    P,
    streaming_solution,
    rho=rho,
    mu=mu,
):
    fR = compliant_first_order(
        np.array([R]), R, Omega, k, P, rho, mu
    )
    lambda_C = fR["lambda_C"]
    C_psi = fR["C_psi"]
    eta_hat = fR["eta_hat"]

    Acoef = k*P/(Omega*rho)

    uz_R = fR["uz"][0]
    ur_R = fR["ur"][0]
    uz_r_R = fR["uz_r"][0]
    ur_r_R = fR["ur_r"][0]

    uz_rr_R = (
        Acoef*k**2*0.5*(iv(0, k*R)+iv(2, k*R))
        - C_psi*lambda_C**3*0.5*(
            jv(0, lambda_C*R)-jv(2, lambda_C*R)
        )
    )

    sigma_zr_r_hat = mu*(
        uz_rr_R + 1j*k*ur_r_R
    )

    sigma_rr_minus_sigma_zz_hat = (
        2.0*mu*(ur_r_R - 1j*k*uz_R)
    )

    gradient = -mu*streaming_solution["u2_r_wall"]

    wall_shift = 0.5*np.real(
        np.conj(eta_hat)*sigma_zr_r_hat
    )

    wall_slope = 0.5*np.real(
        np.conj(1j*k*eta_hat)
        * sigma_rr_minus_sigma_zz_hat
    )

    total = gradient + wall_shift + wall_slope

    return {
        "reference_radius_gradient": gradient,
        "wall_shift": wall_shift,
        "wall_slope": wall_slope,
        "total": total,
    }

traction_caseC = caseC_traction_terms(
    R_caseC,
    Omega_caseC,
    k_caseC,
    P_caseC,
    stream4000,
)

traction_df = pd.DataFrame([traction_caseC])
traction_df.to_csv(
    DATA_DIR / "ch06_caseC_traction_terms.csv",
    index=False
)
display(traction_df)

# Evaluate the complete moving-interface traction at every refinement level.
traction_refinement_rows = []
for Ns in Ns_values:
    terms = caseC_traction_terms(
        R_caseC,
        Omega_caseC,
        k_caseC,
        P_caseC,
        streaming_solutions[Ns],
    )
    traction_refinement_rows.append({
        "N_s": Ns,
        **terms,
        "book_total_Pa": book_tau[Ns],
        "total_relative_error": abs(terms["total"]-book_tau[Ns])/abs(book_tau[Ns]),
    })

traction_refinement_df = pd.DataFrame(traction_refinement_rows)
traction_refinement_df.to_csv(
    DATA_DIR / "ch06_caseC_traction_refinement.csv", index=False
)
display(traction_refinement_df)
assert traction_refinement_df.iloc[-1]["total_relative_error"] < 5e-6


In [ ]:
# Traction decomposition: the moving-interface corrections dominate Case C.
labels = [
    "reference-radius\ngradient",
    "wall shift",
    "wall slope",
    "complete\ntraction",
]
values = [
    traction_caseC["reference_radius_gradient"],
    traction_caseC["wall_shift"],
    traction_caseC["wall_slope"],
    traction_caseC["total"],
]

fig, ax = plt.subplots(figsize=(6.2, 3.3))
bars = ax.bar(
    np.arange(len(values)),
    values,
    facecolor="white",
    edgecolor=BLACK,
    linewidth=0.9,
)
ax.axhline(0.0, color=LIGHT, linewidth=0.8)
ax.set_xticks(np.arange(len(values)))
ax.set_xticklabels(labels)
ax.set_ylabel(r"Contribution to $\langle\tau_w^{(2)}\rangle$ (Pa)")
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_moving_surface_traction_decomposition")
plt.show()

For Case C, the reference-radius gradient term is extremely small compared with the wall-shift correction. The complete moving-surface expansion is therefore not a cosmetic refinement: omitting the shift and slope terms would change the interpreted wall observable at the same perturbation order.

The benchmark values are approximately

$$
-2.8\times10^{-10},
\qquad
-2.2005\times10^{-4},
\qquad
+1.4360\times10^{-6}\ {\rm Pa},
$$

for the gradient, wall-shift, and wall-slope contributions, giving

$$
\langle\tau_w^{(2)}\rangle
\approx
-2.1862\times10^{-4}\ {\rm Pa}.
$$

# Scaling and mechanism-off limit

Steady streaming first appears at second order,

$$
\langle\mathbf u\rangle
=
\epsilon^2
\langle\mathbf u_2\rangle
+
O(\epsilon^3).
$$

Therefore,

$$
\epsilon\rightarrow0
\quad\Longrightarrow\quad
\epsilon^2
\langle\mathbf u_2\rangle
\rightarrow0.
$$

The same order applies to mean streaming flux and mean wall traction.

A second mechanism-off test removes the quadratic Reynolds-stress forcing while retaining the moving-wall boundary term. A third removes the moving-wall boundary term while retaining the Reynolds-stress forcing. These counterfactuals separate the two $O(\epsilon^2)$ sources of the mean field.

In [ ]:
# Mechanism separation for Case C.
def solve_streaming_counterfactual(
    N_s,
    include_reynolds=True,
    include_wall_bc=True,
):
    r = np.linspace(0.0, R_caseC, N_s)
    dr = r[1]-r[0]
    f1 = compliant_first_order(
        r,
        R_caseC,
        Omega_caseC,
        k_caseC,
        P_caseC,
    )

    uz = f1["uz"]
    ur = f1["ur"]
    uz_r = f1["uz_r"]
    eta_hat = f1["eta_hat"]

    mean_convective_z = 0.5*np.real(
        np.conj(ur)*uz_r
        + np.conj(uz)*(1j*k_caseC*uz)
    )

    rhs = (
        rho*mean_convective_z/mu
        if include_reynolds
        else np.zeros_like(r)
    )

    A = lil_matrix((N_s, N_s), dtype=float)
    A[0, 0] = -4.0/dr**2
    A[0, 1] = +4.0/dr**2

    for i in range(1, N_s-1):
        ri = r[i]
        A[i, i-1] = 1.0/dr**2 - 1.0/(2.0*ri*dr)
        A[i, i]   = -2.0/dr**2
        A[i, i+1] = 1.0/dr**2 + 1.0/(2.0*ri*dr)

    wall_value = (
        -0.5*np.real(np.conj(eta_hat)*uz_r[-1])
        if include_wall_bc
        else 0.0
    )

    A[-1, -1] = 1.0
    rhs[-1] = wall_value

    u2 = spsolve(A.tocsr(), rhs)
    Q = 2.0*np.pi*np.trapezoid(r*u2, r)

    return r, u2, Q

counter_specs = [
    ("full", True, True),
    ("Reynolds forcing only", True, False),
    ("moving-wall boundary only", False, True),
    ("both removed", False, False),
]

counter_rows = []
fig, ax = plt.subplots(figsize=(6.0, 3.4))

styles = [
    (BLACK, "-"),
    (DARK, "--"),
    (MID, "-."),
    (LIGHT, ":"),
]

for (label, reynolds_on, wall_on), (gray, ls) in zip(
    counter_specs, styles
):
    rr, uu, qq = solve_streaming_counterfactual(
        1500,
        include_reynolds=reynolds_on,
        include_wall_bc=wall_on,
    )
    counter_rows.append({
        "case": label,
        "Q_stream_m3_s": qq,
    })
    ax.plot(
        rr/R_caseC, uu,
        color=gray, linestyle=ls, label=label
    )

ax.set_xlabel(r"Normalized radius, $r/R$")
ax.set_ylabel(r"$\langle u_{2z}\rangle$ (m s$^{-1}$)")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_streaming_counterfactuals")
plt.show()

counter_df = pd.DataFrame(counter_rows)
counter_df.to_csv(
    DATA_DIR / "ch06_streaming_counterfactuals.csv",
    index=False
)
display(counter_df)

The full mean response is the result of the complete second-order closure, not of one term in isolation.

Turning off the bulk Reynolds-stress forcing removes that **bulk streaming source**, but the Taylor-expanded moving-wall boundary term is itself quadratic in first-order quantities and can still produce a nonzero mean-field contribution. Only when both $O(\epsilon^2)$ sources are removed does this reduced mean problem collapse to zero. This distinction keeps the numerical counterfactual consistent with the chapter's statement that the Reynolds-stress mechanism vanishes when its quadratic forcing is removed, without incorrectly claiming that every possible second-order boundary contribution must vanish with it.

# Parameterized finite-$k$ susceptibility

The Case C benchmark uses $\widehat k=kR=0.2$. To expose how the compliant traveling-wave kinematics depend on axial phase variation, the notebook sweeps $\widehat k$ while holding $R$, $\Omega$, and $P$ fixed at the Case C values.

For each $\widehat k$, the notebook computes:

- $\widehat\eta$ from axial no slip and kinematic compatibility;
- the compatible normal-load ratio;
- the second-order streaming flux.

This is a **controlled reduced-model sweep**. It is not an arterial distribution of physical wavenumber.

In [ ]:
# Controlled kR sweep around the canonical benchmark.
kR_sweep = np.geomspace(0.04, 0.6, 28)
k_sweep_rows = []

for khat in kR_sweep:
    kval = khat/R_caseC
    first = compliant_first_order(
        np.array([R_caseC]),
        R_caseC,
        Omega_caseC,
        kval,
        P_caseC,
    )
    eta_hat = first["eta_hat"]
    Zc = compatible_wall_impedance(
        R_caseC, Omega_caseC, kval, P_caseC
    )

    stream = solve_streaming_case(
        700,
        R_caseC,
        Omega_caseC,
        kval,
        P_caseC,
    )

    k_sweep_rows.append({
        "kR": khat,
        "abs_eta_over_R": abs(eta_hat)/R_caseC,
        "abs_Zcompat_Pa_per_m": abs(Zc),
        "Q_stream_m3_s": stream["Q_stream"],
    })

k_sweep_df = pd.DataFrame(k_sweep_rows)
k_sweep_df.to_csv(
    DATA_DIR / "ch06_kR_sensitivity.csv",
    index=False
)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1))

axes[0].loglog(
    k_sweep_df["kR"],
    k_sweep_df["abs_eta_over_R"],
    color=BLACK
)
axes[0].axvline(kR_caseC, color=LIGHT, linestyle=":", linewidth=1.0)
axes[0].set_xlabel(r"Dimensionless axial wavenumber, $\widehat k=kR$")
axes[0].set_ylabel(r"$|\widehat\eta|/R$")
clean_axes(axes[0])

axes[1].semilogx(
    k_sweep_df["kR"],
    k_sweep_df["Q_stream_m3_s"],
    color=BLACK
)
axes[1].axvline(kR_caseC, color=LIGHT, linestyle=":", linewidth=1.0)
axes[1].set_xlabel(r"Dimensionless axial wavenumber, $\widehat k=kR$")
axes[1].set_ylabel(r"$\langle Q_{\mathrm{stream}}\rangle$ (m$^3$ s$^{-1}$)")
clean_axes(axes[1])

fig.tight_layout(w_pad=1.3)
save_figure(fig, "ch06_kR_sensitivity")
plt.show()

The $\widehat k$ sweep exposes a model dependence that the single benchmark cannot show. It should not be interpreted as a measured arterial wavenumber distribution.

A future physiological use of this dependence would require a clearly defined wave-speed or phase-propagation measurement from which $k$ could be estimated consistently.

# VascuQuest exploration: wall-motion amplitude

The area waveform directly contains moving-boundary information. For every subject/site pair with a valid area waveform, compute

$$
R(t)=\sqrt{\frac{A(t)}{\pi}},
$$

$$
R=\langle R(t)\rangle_t,
$$

and

$$
\epsilon_{\mathrm{VQ}}
=
\frac{\max_t|R(t)-R|}{R}.
$$

This is the cleanest VascuQuest quantity for Chapter 6 because it measures the small-wall-motion ordering parameter without requiring a wall constitutive inversion.

In [ ]:
# Population wall-motion amplitude from PWDB luminal area.
meta = subject_meta.set_index("subject_id")
wall_motion_rows = []
radius_waveforms = {}

for site in SITES:
    print("Processing", SITE_LABELS[site])
    ids_a, A_matrix = load_waveform_matrix(site, "A")

    for sid, A_row in zip(ids_a, A_matrix):
        if sid not in meta.index:
            continue

        A_values = active_prefix(A_row)
        if A_values is None or len(A_values) < 16:
            continue
        R_t = np.sqrt(A_values/np.pi)
        R_mean = float(np.mean(R_t))
        eta_t = R_t - R_mean
        epsilon_vq = float(
            np.max(np.abs(eta_t))/R_mean
        )

        heart_rate = float(meta.loc[sid, "heart_rate_bpm"])
        age = float(meta.loc[sid, "age_years"])
        Omega = 2.0*np.pi*heart_rate/60.0
        alpha = R_mean*np.sqrt(Omega/nu)

        wall_motion_rows.append({
            "subject_id": sid,
            "age_years": age,
            "site": site,
            "R_m": R_mean,
            "heart_rate_bpm": heart_rate,
            "alpha": alpha,
            "epsilon_VQ": epsilon_vq,
        })

        if sid == representative_subject:
            radius_waveforms[(site, sid)] = R_t

wall_motion_df = pd.DataFrame(wall_motion_rows)
wall_motion_df.to_csv(
    DATA_DIR / "ch06_vascuquest_wall_motion.csv",
    index=False
)

groups = [
    wall_motion_df.loc[
        wall_motion_df["site"] == site,
        "epsilon_VQ"
    ].dropna().to_numpy()
    for site in SITES
]

fig, ax = plt.subplots(figsize=(6.6, 3.4))
ax.boxplot(
    groups,
    labels=[SITE_LABELS[s] for s in SITES],
    showfliers=False,
    whis=(5, 95),
    widths=0.58,
    medianprops={"color": BLACK, "linewidth": 1.3},
    boxprops={"color": BLACK, "linewidth": 0.9},
    whiskerprops={"color": DARK, "linewidth": 0.8},
    capprops={"color": DARK, "linewidth": 0.8},
)
ax.tick_params(axis="x", rotation=52)
ax.set_ylabel(r"Observed wall-motion ratio, $\epsilon_{\mathrm{VQ}}$")
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_vascuquest_wall_motion_population")
plt.show()

The population plot is an observation in the PWDB virtual population. It gives a direct sense of how small the moving-boundary amplitude is relative to the local mean radius.

It does not establish that the perturbation expansion is uniformly accurate for every subject/site pair. That requires a separate error assessment, especially for the upper tail of $\epsilon_{\mathrm{VQ}}$.

# Pressure--area phase and loop geometry

A compliant wall is a dynamical boundary, so amplitude alone is insufficient. The phase relation between pressure and area is also informative.

For the representative subject and selected arterial sites, the notebook therefore plots the pressure--area loop over one cycle and computes the first-harmonic phase difference.

This remains a waveform observation. A pressure--area loop by itself does not uniquely identify the complete Kelvin--Voigt impedance because the Chapter 6 normal-load balance also contains fluid normal stress, wall tension, wall inertia, and finite-$k$ coupling.

In [ ]:
# Representative pressure-area loops.
REP_SITES = ["AorticRoot", "Carotid", "Femoral", "Radial"]

fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.4))
axes = axes.ravel()

phase_rows = []

for ax, site in zip(axes, REP_SITES):
    ids_p, P_matrix = load_waveform_matrix(site, "P")
    ids_a, A_matrix = load_waveform_matrix(site, "A")
    assert np.array_equal(ids_p, ids_a)

    idx = np.where(ids_p == representative_subject)[0]
    if len(idx) != 1:
        raise RuntimeError(
            f"Representative subject not found uniquely at {site}."
        )
    idx = int(idx[0])

    p = active_prefix(P_matrix[idx])
    area = active_prefix(A_matrix[idx])
    if p is None or area is None:
        raise RuntimeError(f"Internal missing samples in representative waveform at {site}.")
    n = min(len(p), len(area))
    p, area = p[:n], area[:n]
    if n < 16:
        raise RuntimeError(f"Too few active samples at {site}.")

    p1 = first_harmonic(p)
    A1 = first_harmonic(area)
    if abs(p1) < 1e-12 or not np.isfinite(A1):
        raise RuntimeError(f"Invalid first harmonic at {site}.")
    phase_difference = np.angle(A1/p1)

    phase_rows.append({
        "site": site,
        "pressure_area_phase_rad": phase_difference,
    })

    ax.plot(p, area*1e6, color=BLACK)
    ax.set_xlabel(r"$p$ (mmHg)")
    ax.set_ylabel(r"$A$ (mm$^2$)")
    ax.set_title(SITE_LABELS[site])
    clean_axes(ax)

fig.tight_layout()
save_figure(fig, "ch06_representative_pressure_area_loops")
plt.show()

phase_df = pd.DataFrame(phase_rows)
phase_df.to_csv(
    DATA_DIR / "ch06_representative_pressure_area_phase.csv",
    index=False
)
display(phase_df)

The loop area and orientation reflect a phase-dependent pressure--area relation in the virtual data. The Chapter 6 wall law provides one reduced mechanical language for such phase dependence, but the notebook does not equate the loop directly with $K_s\tau_v$ or any other single coefficient.

In [ ]:
# Population first-harmonic pressure-area phase at selected sites.
phase_population_rows = []

for site in REP_SITES:
    print("Phase population:", SITE_LABELS[site])
    ids_p, P_matrix = load_waveform_matrix(site, "P")
    ids_a, A_matrix = load_waveform_matrix(site, "A")
    assert np.array_equal(ids_p, ids_a)

    for sid, p_row, a_row in zip(ids_p, P_matrix, A_matrix):
        if sid not in meta.index:
            continue

        p = active_prefix(p_row)
        area = active_prefix(a_row)
        if p is None or area is None:
            continue
        n = min(len(p), len(area))
        p, area = p[:n], area[:n]
        if n < 16:
            continue

        p1 = first_harmonic(p)
        A1 = first_harmonic(area)
        if (not np.isfinite(p1.real) or not np.isfinite(p1.imag) or
                not np.isfinite(A1.real) or not np.isfinite(A1.imag) or
                abs(p1) < 1e-12 or abs(A1) < 1e-18):
            continue

        dphi = np.angle(A1/p1)

        phase_population_rows.append({
            "subject_id": sid,
            "age_years": float(meta.loc[sid, "age_years"]),
            "site": site,
            "pressure_area_phase_rad": dphi,
            "pressure_area_phase_deg": np.degrees(dphi),
        })

phase_population_df = pd.DataFrame(phase_population_rows)
phase_population_df.to_csv(
    DATA_DIR / "ch06_pressure_area_phase_population.csv",
    index=False
)

age_phase_rows = []
for (age, site), group in phase_population_df.groupby(["age_years", "site"]):
    age_phase_rows.append({
        "age_years": age,
        "site": site,
        "pressure_area_phase_rad": phase_aware_median(
            group["pressure_area_phase_rad"].to_numpy()
        ),
        "count": int(len(group)),
    })
age_phase = pd.DataFrame(age_phase_rows)

styles = [
    (BLACK, "-", "o"),
    (DARK, "--", "s"),
    (MID, "-.", "^"),
    (LIGHT, ":", "D"),
]

fig, ax = plt.subplots(figsize=(6.0, 3.35))

for site, (gray, ls, marker) in zip(REP_SITES, styles):
    sub = age_phase.loc[age_phase["site"] == site]
    ax.plot(
        sub["age_years"],
        sub["pressure_area_phase_rad"],
        color=gray,
        linestyle=ls,
        marker=marker,
        markersize=4,
        label=SITE_LABELS[site],
    )

ax.set_xlabel("PWDB source age (years)")
ax.set_ylabel("Median first-harmonic pressure--area phase (rad)")
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_age_group_pressure_area_phase")
plt.show()

The age-group phase comparison is descriptive. It does not establish that age changes a specific Kelvin--Voigt coefficient.

It shows only that the virtual population contains systematic variation in the local pressure--area phase relation, which is relevant context for a chapter whose central new ingredient is a dynamical moving boundary.

The reported group median is evaluated with phase-aware recentering so values near $-\pi$ and $+\pi$ are not treated as artificially far apart.

# Relating observed wall motion to the perturbation ordering

The perturbation result

$$
\langle\mathbf u\rangle
=
\epsilon^2\langle\mathbf u_2\rangle
+
O(\epsilon^3)
$$

implies that mean transport is quadratically sensitive to the small wall-motion amplitude.

The notebook therefore uses the VascuQuest distribution of $\epsilon_{\mathrm{VQ}}$ only to show the **relative second-order scaling**

$$
\left(
\frac{\epsilon_{\mathrm{VQ}}}
{\epsilon_{\mathrm{VQ,ref}}}
\right)^2.
$$

This comparison does not assign the same dimensional $\langle\mathbf u_2\rangle$ coefficient to every artery. It isolates the perturbation-order effect of wall-motion amplitude alone.

In [ ]:
# Relative second-order scaling from observed VascuQuest wall motion.
epsilon_ref = float(
    wall_motion_df["epsilon_VQ"].median()
)
wall_motion_df["relative_second_order_scaling"] = (
    wall_motion_df["epsilon_VQ"]/epsilon_ref
)**2

fig, ax = plt.subplots(figsize=(5.9, 3.3))
ax.scatter(
    wall_motion_df["epsilon_VQ"],
    wall_motion_df["relative_second_order_scaling"],
    s=8,
    facecolors="none",
    edgecolors=MID,
    linewidths=0.5,
)
xline = np.linspace(
    wall_motion_df["epsilon_VQ"].min(),
    wall_motion_df["epsilon_VQ"].max(),
    300,
)
ax.plot(
    xline,
    (xline/epsilon_ref)**2,
    color=BLACK,
)
ax.set_xlabel(r"Observed wall-motion ratio, $\epsilon_{\mathrm{VQ}}$")
ax.set_ylabel("Relative $O(\epsilon^2)$ scaling")
clean_axes(ax)
fig.tight_layout()

save_figure(fig, "ch06_vascuquest_second_order_scaling")
plt.show()

This figure is a direct consequence of perturbation order, not a population prediction of streaming flux.

Two arteries with the same $\epsilon_{\mathrm{VQ}}$ can still have different second-order transport because the coefficient multiplying $\epsilon^2$ depends on the first-order mode, finite-$k$ structure, frequency, radius, and wall coupling.

# Optional higher-order envelope branch

At $O(\epsilon^3)$, a solvability condition can produce the complex Ginzburg--Landau equation

$$
\frac{\partial A}{\partial\mathcal T}
=
\sigma A
+
\xi\frac{\partial^2A}{\partial Z^2}
-
\beta|A|^2A.
$$

Chapter 6 and Chapter 8 deliberately stop the verified compliance workflow at the second-order streaming problem.

This notebook therefore does **not** manufacture numerical values for

$$
\sigma,\qquad\xi,\qquad\beta,
$$

or for the Benjamin--Feir--Newell diagnostic.

Those coefficients require the complete direct mode, adjoint mode, second-order mean and second harmonic, resonant cubic forcing, normalization, refinement, and denominator-conditioning checks described in Appendix J.

# What the reader should learn

1. **Compliance changes the boundary condition, not merely a coefficient in the rigid solution.** The finite-$k$ first-order mode contains radial motion and a dynamical wall displacement.

2. **The pressure field is not radially uniform at finite $k$.** The regular pressure amplitude is $P I_0(kr)$.

3. **The wall impedance has separable physical contributions.** Elastic stiffness, Kelvin--Voigt phase lag, axial tension, and wall inertia enter with different frequency and wavenumber dependence.

4. **The classical Womersley solution is a limit of the compliant formulation.** It is recovered as the wall becomes rigid and $k\to0$ with the axial pressure-gradient amplitude held finite.

5. **Steady streaming is a zero-frequency quadratic response.** The first-order field has zero mean, but its self-interaction does not.

6. **The moving-wall boundary correction is part of the $O(\epsilon^2)$ problem.** It cannot be replaced by homogeneous mean no slip.

7. **Mean flux and mean wall traction are distinct observables.** The latter must be evaluated on the moving interface and contains shift and slope corrections.

8. **The Case C benchmark is a verification problem, not a calibrated arterial wall.**

9. **VascuQuest directly informs wall-motion amplitude and pressure--area phase.** It does not uniquely identify $K_s$, $\tau_v$, $T_w$, or $\rho_wh_w$ without an explicit inverse model and additional information.

10. **The perturbation order constrains interpretation.** A second-order mean response must vanish quadratically as $\epsilon\to0$.

# Chapter-enrichment candidates

The notebook produces ten principal figures.

**Candidate 1 — Case C first-order axial and radial mode.**  
Potential book candidate if the chapter needs a direct visual demonstration that wall compliance opens radial fluid motion at first order.

**Candidate 2 — long-wave Womersley recovery.**  
Strong verification figure but probably notebook-only because the limiting argument is already explicit in the chapter.

**Candidate 3 — reproduced Case C streaming profile.**  
Already present in Chapter 6. Use only as a replacement if the notebook rendering is superior.

**Candidate 4 — moving-surface traction decomposition.**  
Strong book candidate. It shows immediately why the wall-shift and wall-slope terms cannot be omitted.

**Candidate 5 — streaming mechanism counterfactuals.**  
Strong book candidate if the chapter benefits from separating Reynolds-stress forcing from the inhomogeneous moving-wall boundary condition.

**Candidate 6 — $\widehat k$ sensitivity.**  
Notebook-first result. It is useful for understanding finite-$k$ dependence but should not enter the book without a clear reason because $\widehat k$ is not yet physiologically calibrated.

**Candidate 7 — VascuQuest wall-motion population.**  
Strong potential book candidate. It gives direct population context for the chapter's organizing small parameter without fitting a wall law.

**Candidate 8 — representative pressure--area loops.**  
Potentially strong book figure because it provides an intuitive visual signature of a dynamical compliant boundary.

**Candidate 9 — age-group pressure--area phase.**  
Primarily notebook material unless execution reveals a clear, nonredundant pattern.

**Candidate 10 — VascuQuest relative $O(\epsilon^2)$ scaling.**  
Notebook-first. It illustrates perturbation order but must not be presented as a streaming-flux prediction.

No figure is promoted automatically. Existing Chapter 6 figures should be replaced rather than duplicated when the notebook-derived version serves the same scientific purpose better.

In [ ]:
# Reproducibility record.
manifest = {
    "book": "Nonlinear Arterial Hemodynamics",
    "chapter": 6,
    "chapter_title": "Wall Compliance and Mean Transport",
    "vascuquest_git_ref": VQ_GIT_REF,
    "pwdb_record_id": PWDB_RECORD_ID,
    "pwdb_doi": PWDB_DOI,
    "rho_kg_m3": rho,
    "mu_Pa_s": mu,
    "nu_m2_s": nu,
    "caseC_R_m": R_caseC,
    "caseC_f_Hz": f_caseC,
    "caseC_kR": kR_caseC,
    "caseC_P_Pa": P_caseC,
    "caseC_alpha": alpha_caseC,
    "caseC_eta_hat_m": {
        "real": float(eta_caseC.real),
        "imag": float(eta_caseC.imag),
    },
    "caseC_Zcompat_Pa_per_m": {
        "real": float(Zcompat_caseC.real),
        "imag": float(Zcompat_caseC.imag),
    },
    "caseC_Q_stream_N4000_m3_s": float(
        stream4000["Q_stream"]
    ),
    "caseC_tau_w2_Pa": float(
        traction_caseC["total"]
    ),
    "representative_subject_id": representative_subject,
    "representative_age_years": float(target_age),
    "sites": SITES,
    "source_age_strata_years": [float(x) for x in source_ages],
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "execution_status": "completed to this cell",
    "mean_flow_numerical_method": (
        "independent second-order centered interior radial operator; "
        "fourth-order backward wall derivative"
    ),
    "chapter8_caseC_finest_flux_relative_error": float(
        streaming_df.iloc[-1]["Q_relative_error"]
    ),
    "chapter8_caseC_finest_traction_relative_error": float(
        traction_refinement_df.iloc[-1]["total_relative_error"]
    ),
    "qualification": (
        "Case C is a dimensional verification problem with normalized P=1 Pa. "
        "PWDB supplies wall-motion and pressure-area waveform context. "
        "Kelvin-Voigt coefficients are not inferred from PWDB unless an explicit "
        "inverse model and independent parameter definitions are introduced."
    ),
}
(META_DIR / "reproducibility_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("\nGenerated PDF figures:")
for path in sorted(FIG_DIR.glob("*.pdf")):
    print(" -", path.name)

# Reproducibility record

A successful **Run all** execution writes:

- B&W vector PDF figures and high-resolution PNG previews;
- Case C first-order benchmark values;
- long-wave rigid-Womersley recovery data;
- an independent radial streaming-refinement table with explicit errors relative to the Chapter 8 benchmark;
- complete moving-surface traction refinement and decomposition;
- second-order mechanism-off counterfactuals;
- controlled $\widehat k$ sensitivity data;
- VascuQuest wall-motion and pressure--area phase population tables;
- deterministic representative-subject metadata;
- VascuQuest/PWDB verification metadata;
- a final reproducibility manifest.

Waveform rows containing internal missing samples are rejected rather than compressed, so cardiac-cycle phase is never changed silently. The released notebook contains no hidden wall-law fit, no manual subject selection, and no numerical CGL coefficients unsupported by the complete third-order solvability calculation.